In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

# Date window: 8 Apr 2026 to 14 Apr 2026 (inclusive)
start_date = pd.Timestamp("2026-04-08 00:00:00")
end_date = pd.Timestamp("2026-04-14 23:45:00")


def get_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "raw").is_dir() and (p / "data" / "predictions").is_dir():
            return p
    return here


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mape_pct(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    non_zero_mask = y_true != 0
    if not np.any(non_zero_mask):
        return np.nan
    return float(np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100)


def r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot == 0:
        return np.nan
    return float(1 - (ss_res / ss_tot))


# Project paths
repo_root = get_repo_root()
predictions_dir = repo_root / "data" / "predictions"
actual_path = repo_root / "data" / "raw" / "iex-dam-0201-0421.csv"

# Load and filter actuals
actual_df = pd.read_csv(actual_path, parse_dates=["period_start"]).rename(columns={"period_start": "timestamp"})
actual_df = actual_df.loc[(actual_df["timestamp"] >= start_date) & (actual_df["timestamp"] <= end_date), ["timestamp", "purchase_bid", "sell_bid"]]

# Evaluate each prediction file
rows = []
for pred_path in sorted(predictions_dir.glob("*_0408_0414.csv")):
    name = pred_path.stem  # e.g. pb_lstm_0408_0414
    side_code, model_name, *_ = name.split("_")
    target_col = "purchase_bid" if side_code == "pb" else "sell_bid"

    pred_df = pd.read_csv(pred_path)
    # Handle files with an unnamed first index column.
    if "timestamp" not in pred_df.columns and pred_df.shape[1] >= 3:
        pred_df = pred_df.iloc[:, 1:]

    pred_df["timestamp"] = pd.to_datetime(pred_df["timestamp"])
    pred_df = pred_df[["timestamp", "pred"]].rename(columns={"pred": "y_pred"})

    merged = actual_df[["timestamp", target_col]].rename(columns={target_col: "y_true"}).merge(pred_df, on="timestamp", how="inner")

    y_true = merged["y_true"].to_numpy(dtype=float)
    y_pred = merged["y_pred"].to_numpy(dtype=float)

    rows.append(
        {
            "file": pred_path.name,
            "model": model_name,
            "target": target_col,
            "n_points": len(merged),
            "mae": mae(y_true, y_pred),
            "rmse": rmse(y_true, y_pred),
            "mape_pct": mape_pct(y_true, y_pred),
            "r2": r2(y_true, y_pred),
        }
    )

results_df = pd.DataFrame(rows).sort_values(["target", "model"]).reset_index(drop=True)
results_df

,file,model,target,n_points,mae,rmse,mape_pct,r2
0,pb_boost_0408_0414.csv,boost,purchase_bid,672,2230.389866,2678.726215,16.905051,0.063306
1,pb_lstm_0408_0414.csv,lstm,purchase_bid,672,2687.302945,3348.395277,21.629690,-0.463574
2,pb_timeseries_0408_0414.csv,timeseries,purchase_bid,672,2667.719361,3210.480092,18.552311,-0.345492
3,sb_boost_0408_0414.csv,boost,sell_bid,672,4661.100218,6079.452307,22.142887,0.896978
4,sb_lstm_0408_0414.csv,lstm,sell_bid,672,8815.989282,11777.062402,32.957818,0.613388
5,sb_timeseries_0408_0414.csv,timeseries,sell_bid,672,4848.497937,6029.953587,37.143832,0.898649
